# CS2 player shape training (Kaggle)

Train YOLO-seg for CS2 player silhouettes (class `player`).

Settings: Accelerator → GPU, Internet → On, then Run All.

Free-tier path when the network is available: data → bootstrap shapes → train →
self-label → retrain for N iterations.

Default data: RekaAI/CS2-10k (auto-download 1 shard). Optional: Add Input zip.
Outputs under `/kaggle/working/cs2_data` (Output tab).

Validation defaults to held-out train/val (`allow_leaky_val=False`).
Profile `cloud_t4`, `resume=True`. Knobs: teacher gate, conf schedule,
CS2-10k holdout fraction, bootstrap. Optional leaky val via `ALLOW_LEAKY_VAL=1`.

Labels: default bootstrap is COCO person mapped to `player`. Set
`USE_EDGESAM=True` for Vombit+EdgeSAM teachers (downloads more ONNX assets;
more disk and time).


In [ ]:
# @title 1) Install + import (autonomous)
import os
import subprocess
import sys
from pathlib import Path


def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core training + export deps
_pip(
    "ultralytics",
    "onnx",
    "onnxconverter-common",
    "opencv-python-headless",
    "huggingface_hub",
    "matplotlib",
    "pyyaml",
)
try:
    import torch  # noqa: F401
except Exception:
    _pip("torch")
try:
    _pip("onnxruntime-gpu")
except Exception as e:
    print("onnxruntime-gpu optional:", e)

def _try_import() -> bool:
    try:
        import cs2_vision_access.workflows.training.remote_autonomous  # noqa: F401
        return True
    except Exception:
        return False

# 1) Optional: pip install from CS2_VISION_REPO (git+https://... or path)
REPO = os.environ.get("CS2_VISION_REPO", "").strip()
if REPO:
    print("Installing package from CS2_VISION_REPO:", REPO)
    try:
        _pip(REPO)
    except Exception as e:
        print("CS2_VISION_REPO install failed:", e)

# 2) Editable / sys.path local clone under platform roots

_CLONE_ROOTS = [Path("/kaggle/working"), Path("/kaggle/input"), Path(".")]
_LOCAL_CANDIDATES = [
    Path("/kaggle/working/computer-vision-accessibility"),
    Path("/kaggle/input/computer-vision-accessibility"),
    Path("computer-vision-accessibility"),
    Path("/kaggle/working"),
    Path("."),
]
for cand in _LOCAL_CANDIDATES:
    if (cand / "pyproject.toml").is_file() and (cand / "src" / "cs2_vision_access").is_dir():
        print("Editable/local package:", cand)
        try:
            _pip("-e", str(cand))
        except Exception:
            sys.path.insert(0, str(cand / "src"))
        break

for root in _CLONE_ROOTS:
    src = root / "src"
    if (src / "cs2_vision_access").is_dir():
        p = str(src)
        if p not in sys.path:
            sys.path.insert(0, p)
    pkg = root / "computer-vision-accessibility"
    if (pkg / "pyproject.toml").is_file() and (pkg / "src" / "cs2_vision_access").is_dir():
        try:
            _pip("-e", str(pkg))
        except Exception:
            p = str(pkg / "src")
            if p not in sys.path:
                sys.path.insert(0, p)

# 3) If still not importable: shallow git clone + pip -e / sys.path
if not _try_import():
    git_url = os.environ.get("CS2_VISION_GIT", "").strip()
    if not git_url:
        git_url = "https://github.com/xai-org/computer-vision-accessibility.git"
    dest = Path("/kaggle/working/computer-vision-accessibility")
    if not (dest / "src" / "cs2_vision_access").is_dir():
        print("Cloning package (depth=1):", git_url, "→", dest)
        try:
            subprocess.check_call(
                ["git", "clone", "--depth", "1", git_url, str(dest)],
            )
        except Exception as e:
            raise SystemExit(
                f"git clone failed ({e}).\n"
                "Set CS2_VISION_REPO=git+https://github.com/<you>/computer-vision-accessibility.git\n"
                "or clone repo into /kaggle/working and re-run"
            ) from e
    try:
        _pip("-e", str(dest))
    except Exception:
        sys.path.insert(0, str(dest / "src"))

# Fail fast if the autonomous entry point is still missing
try:
    from cs2_vision_access.workflows.training.remote_autonomous import (
        resolve_remote_dataset_zip,
        run_autonomous_loop,
    )
except Exception as e:
    raise SystemExit(
        "run_autonomous_loop still not importable after install/clone.\n"
        f"  last error: {e}\n"
        "  Set CS2_VISION_REPO=git+https://github.com/<you>/computer-vision-accessibility.git\n"
        "  or clone the repo into the working directory and re-run this cell."
    ) from e

import torch

print("remote_autonomous ready")
print("CUDA:", torch.cuda.is_available(), end=" ")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print(
        "\nWarning: no GPU detected; training will be slow. "
        "For Colab: Runtime → Change runtime type → T4 GPU. "
        "Smoke runs on CPU are allowed."
    )


In [ ]:
# @title 2) Config + CS2-10k / zip discovery
import os
from pathlib import Path

# Target class: **player** silhouettes / shapes (PRODUCT_CLASSES {0: player})

DATASET_ZIP = os.environ.get(
    "CS2_DATASET_ZIP",
    "/kaggle/input/cs2-training-data/cs2_train_bundle.zip",
)

# --- shape-training defaults ---
ITERATIONS = 4  # @param {type:"slider", min:1, max:8, step:1}
EPOCHS_PER_ITER = 0  # @param {type:"integer"}  # 0 = profile cloud_t4 epochs
CONF_SELF_TRAIN = 0.45  # @param {type:"number"}
CONF_SCHEDULE = True  # @param {type:"boolean"}
USE_TEACHER_GATE = True  # @param {type:"boolean"}  # multi-iter teacher = prev best
CS2_10K_HOLDOUT_FRAC = 0.2  # @param {type:"number"}
USE_CS2_10K = True  # @param {type:"boolean"}
BOOTSTRAP = True  # @param {type:"boolean"}
USE_EDGESAM = False  # @param {type:"boolean"}  # opt-in Vombit+EdgeSAM auto-label
MIN_LABEL_RATIO = 0.05  # @param {type:"number"}

CS2_10K_MAPS = "mirage,dust2"  # @param {type:"string"}
CS2_10K_MAX_SHARDS = 1  # @param {type:"integer"}
CS2_10K_MAX_VIDEOS = 20  # @param {type:"integer"}
CS2_10K_FRAMES_PER_VIDEO = 10  # @param {type:"integer"}

BATCH = 16  # @param {type:"slider", min:2, max:32, step:2}
IMGSZ = 416  # @param {type:"integer"}
BASE_MODEL = "yolo11n-seg.pt"  # @param ["yolo11n-seg.pt", "yolo11s-seg.pt"]
OUTPUT_DIR = "/kaggle/working/cs2_data"  # @param {type:"string"}
# Held-out val default; set ALLOW_LEAKY_VAL=1 for flat train=val research only.
ALLOW_LEAKY_VAL = os.environ.get("ALLOW_LEAKY_VAL", "").strip().lower() in (
    "1", "true", "yes",
)  # @param {type:"boolean"}

# --- EdgeSAM auto-label (opt-in Vombit+EdgeSAM) ---
USE_EDGESAM = False  # @param {type:"boolean"}  # opt-in: Vombit+EdgeSAM auto-label
DOWNLOAD_EDGESAM = True  # @param {type:"boolean"}  # download teachers if missing
EDGESAM_ARTIFACTS = ""  # @param {type:"string"}  # empty = data_dir/edgesam_assets
EDGESAM_CONF = 0.4  # @param {type:"number"}
EDGESAM_COCO_FALLBACK = True  # @param {type:"boolean"}

zip_path = resolve_remote_dataset_zip(
    DATASET_ZIP if Path(DATASET_ZIP).is_file() else None,
    search_roots=["/kaggle/input", "/kaggle/working", "."],
)
if zip_path is None and not USE_CS2_10K:
    raise FileNotFoundError(
        "No dataset zip under /kaggle/input and USE_CS2_10K=False. "
        "Add Input zip or enable USE_CS2_10K (default)."
    )
print("Target class: player silhouettes / shapes")
print("Dataset zip:", zip_path if zip_path else "(none; RekaAI/CS2-10k)")
print(f"CS2-10k: {USE_CS2_10K} maps={CS2_10K_MAPS}")
print(f"  holdout_video_fraction={CS2_10K_HOLDOUT_FRAC}")
print(
    f"Loop: {ITERATIONS} iters | conf={CONF_SELF_TRAIN} schedule={CONF_SCHEDULE} "
    f"teacher={USE_TEACHER_GATE}"
)
print(f"Bootstrap={BOOTSTRAP} use_edgesam={USE_EDGESAM} min_label_ratio={MIN_LABEL_RATIO}")
print(f"Val: allow_leaky_val={ALLOW_LEAKY_VAL} (default False = held-out split)")
print(
    f"EdgeSAM: use={USE_EDGESAM} download={DOWNLOAD_EDGESAM} conf={EDGESAM_CONF} "
    f"coco_fallback={EDGESAM_COCO_FALLBACK} artifacts={EDGESAM_ARTIFACTS or '(default)'}"
)
if USE_EDGESAM:
    print(
        "USE_EDGESAM=True: multi-ONNX teacher download (Vombit+EdgeSAM); "
        "disk usage and runtime will increase."
    )
print("Profile: cloud_t4 | resume: True")


In [ ]:
# @title 3) Iterative train loop (CS2-10k capable)
import json

from cs2_vision_access.training.contracts import PRODUCT_CLASSES

epochs = None if EPOCHS_PER_ITER in (0, None) else int(EPOCHS_PER_ITER)
maps = tuple(m.strip() for m in CS2_10K_MAPS.split(",") if m.strip())

_loop_kwargs = dict(
    dataset_zip=zip_path,
    data_dir=OUTPUT_DIR,
    iterations=int(ITERATIONS),
    epochs_per_iter=epochs,
    batch=int(BATCH),
    imgsz=int(IMGSZ),
    base_model=BASE_MODEL,
    conf_threshold=float(CONF_SELF_TRAIN),
    conf_schedule=bool(CONF_SCHEDULE),
    use_teacher_gate=bool(USE_TEACHER_GATE),
    bootstrap_if_needed=bool(BOOTSTRAP),
    min_label_ratio=float(MIN_LABEL_RATIO),
    classes=PRODUCT_CLASSES,  # {0: player} shapes
    origin="Autonomous remote player-shape train (CS2-10k / YOLO-seg)",
    install_deps=False,
    allow_leaky_val=bool(ALLOW_LEAKY_VAL),
    profile="cloud_t4",
    resume=True,
    use_cs2_10k=bool(USE_CS2_10K),
    cs2_10k_maps=maps or ("mirage", "dust2"),
    cs2_10k_max_shards=int(CS2_10K_MAX_SHARDS),
    cs2_10k_max_videos=int(CS2_10K_MAX_VIDEOS),
    cs2_10k_frames_per_video=int(CS2_10K_FRAMES_PER_VIDEO),
    cs2_10k_holdout_video_fraction=float(CS2_10K_HOLDOUT_FRAC),
    continue_on_self_train_error=False,
    use_edgesam=bool(USE_EDGESAM),
    download_edgesam=bool(DOWNLOAD_EDGESAM),
    edgesam_artifacts_dir=(EDGESAM_ARTIFACTS or None),
    edgesam_confidence=float(EDGESAM_CONF),
    edgesam_coco_fallback=bool(EDGESAM_COCO_FALLBACK),
)
try:
    report = run_autonomous_loop(**_loop_kwargs)
except TypeError as e:
    # remote_autonomous may not expose EdgeSAM kwargs yet
    print(
        "Note: run_autonomous_loop does not accept EdgeSAM kwargs yet "
        f"({e}); retrying without them."
    )
    for _k in (
        "use_edgesam",
        "download_edgesam",
        "edgesam_artifacts_dir",
        "edgesam_confidence",
        "edgesam_coco_fallback",
    ):
        _loop_kwargs.pop(_k, None)
    report = run_autonomous_loop(**_loop_kwargs)

print("--- autonomous shape train complete ---")
print("status:", report.status)
print("final_label_count:", report.final_label_count)
print("final_image_count:", report.final_image_count)
print("onnx_path:", report.onnx_path)
print("best_onnx_path:", report.best_onnx_path)
print("best_iteration:", report.best_iteration)
print("best_map:", report.best_map)
print("bundle_path:", report.bundle_path)
print("progress_report_md:", report.progress_report_md)
print("dataset_license:", report.dataset_license)
print(
    "Live: cs2-vision live --model <onnx> --manifest <manifest> --class-name player"
)
print(json.dumps(report.as_dict(), indent=2)[:2500])


In [ ]:
# @title 3b) Progress report (shapes / metrics)
from pathlib import Path

from IPython.display import Image, Markdown, display

progress = Path(getattr(report, "progress_dir", None) or Path(report.data_dir) / "progress")
md = Path(getattr(report, "progress_report_md", None) or progress / "report.md")
if md.is_file():
    display(Markdown(md.read_text(encoding="utf-8")[:50000]))
for png in sorted(progress.glob("*.png")):
    display(Image(filename=str(png)))
if not progress.is_dir():
    print("No progress/ yet; check train logs")


In [ ]:
# @title 4) Outputs (Output tab + FileLink)
from pathlib import Path

out = Path(OUTPUT_DIR)
for p in sorted(out.glob("*")):
    if p.is_file():
        print(f"{p.name:40s} {p.stat().st_size/1e6:8.2f} MB")
try:
    from IPython.display import FileLink, display
    for name in (
        "cs2-yolo11n-seg.onnx",
        "cs2-yolo11n-seg-fp16.onnx",
        "cs2-yolo11n-seg.model.json",
        "cs2-yolo11n-seg-bundle.zip",
        "autonomous_report.json",
    ):
        fp = out / name
        if fp.is_file():
            display(FileLink(str(fp)))
except Exception as e:
    print("FileLink:", e)
print("Use: cs2-vision live --model … --manifest … --class-name player")


## Notes

- **Run All** after GPU + Internet (CS2-10k auto-loads; Input zip optional).
- **No human labeling** required for the free-tier path when network works.
- Target class is **`player`** shapes/silhouettes.
- **Held-out val is default** (`allow_leaky_val=False`). `ALLOW_LEAKY_VAL=1`
  for flat train=val only when intentionally research-leaky.
- **Teacher gate**, **conf schedule**, **CS2-10k holdout** exposed in config.
- **EdgeSAM** (`USE_EDGESAM`) opt-in Vombit+EdgeSAM auto-label (default COCO bootstrap).
- Profile **`cloud_t4`**, **`resume=True`**.
- Progress cell shows `progress/report.md` + plots after train.
- Final class is **`player`** (`--class-name player` in live).
- After train, inspect `status`, `final_label_count`, `best_iteration`,
  `best_map`, `bundle_path`, `dataset_license`.
- Package: `CS2_VISION_REPO` or auto-clone via `CS2_VISION_GIT`.
